# IDM Project — Real-Time Object Detection using YOLOv8

**Course:** CSET225 | Intelligent Design Models | Summer Term  
**Student:** Sanskar Singh

This notebook trains a YOLOv8 model to detect objects in images.  
It uses the COCO128 dataset (128 images, 80 object classes like person, car, bus, dog).

## Step 1: Install Required Libraries and Check GPU

In [ ]:
# Install the Ultralytics package — it contains the YOLOv8 model code
!pip install -q ultralytics

# Import PyTorch — the deep learning framework YOLOv8 runs on
import torch

# Check if a GPU is available for faster training
print('CUDA available:', torch.cuda.is_available())

# Print the GPU name so we know what hardware we are using
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## Step 2: Load Pretrained Model and Run a Test Detection

YOLOv8n is the smallest (nano) version of YOLOv8.  
It is already trained on the full COCO dataset (80 classes, 118,000 images).  
We load it and test it on a bus image to confirm it works.

In [ ]:
# Path is a helper class for joining file paths safely
from pathlib import Path

# YOLO is the main class from Ultralytics — used to load, train, and run the model
from ultralytics import YOLO

# Image is from PIL (Python Imaging Library) — used to open and display images
from PIL import Image

# Load the pretrained YOLOv8 nano model (downloads automatically if not present)
# 'yolov8n.pt' = the nano weights file (3.2 million parameters, 6.2 MB)
model = YOLO('yolov8n.pt')

# Run object detection on a sample bus image from the internet
# conf=0.25 means: only show detections with 25% or higher confidence
# save=True means: save the annotated image (with bounding boxes) to disk
results = model.predict('https://ultralytics.com/images/bus.jpg', conf=0.25, save=True)

# results[0].save_dir is the folder where the annotated image was saved
# Path(...) wraps it so we can use / to join the folder path with the filename
# display() shows the annotated image with bounding boxes inside the notebook
display(Image.open(Path(results[0].save_dir) / 'bus.jpg'))


## Step 3: Fine-Tune the Model on COCO128 Dataset

Fine-tuning means: take a model that already knows how to detect objects (pretrained)  
and train it a bit more on a smaller dataset so it adapts.

COCO128 = 128 images from the COCO dataset (auto-downloads by Ultralytics).

Key settings explained:
- **lr0=0.001**: Learning rate = how big each training step is. Small value = careful steps.
- **freeze=10**: Freeze the first 10 layers (backbone) so they keep their pretrained features.
  Only the detection head (last layers) gets trained.
- **epochs=15**: How many times the model sees all 128 images. 15 is enough for a small dataset.

In [ ]:
# Reload a FRESH pretrained model before training
# This is important: the model from Step 2 was already used for prediction
# We need a clean copy of the pretrained weights to start training from
model = YOLO('yolov8n.pt')

# Start fine-tuning (training) the model on the COCO128 dataset
results = model.train(
    data='coco128.yaml',   # Dataset config file — tells the model where images and labels are
    epochs=15,             # Number of training passes through all 128 images
    imgsz=640,             # Resize all images to 640x640 pixels before feeding to the model
    batch=16,              # Process 16 images at a time (one batch) before updating weights
    device=0,              # Use GPU number 0 (the Tesla T4) for training — much faster than CPU
    lr0=0.001,             # Learning rate = 0.001 — small steps so we don't destroy pretrained features
    freeze=10,             # Freeze first 10 layers (backbone) — only train the detection head
    patience=20,           # Stop early if no improvement for 20 epochs (saves time)
    name='train_fresh'     # Name of the folder where training results will be saved
)


## Step 4: Evaluate the Fine-Tuned Model

After training, we test the model on the validation set to measure performance.

Four metrics are printed:
- **mAP50**: Mean Average Precision at IoU 0.50 — how accurate are the detections (loose match)
- **mAP50-95**: Same but averaged across stricter overlap thresholds (the standard COCO metric)
- **Precision**: Of all detections made, what fraction were correct (fewer false alarms)
- **Recall**: Of all real objects in the images, what fraction did the model find (fewer misses)

In [ ]:
# glob searches for files matching a pattern — ** means search all subfolders
import glob

# Find the best.pt file (the model weights from the best epoch during training)
# recursive=True means search inside every subfolder
matches = glob.glob('**/best.pt', recursive=True)

# Use the first match found, or fall back to a default path
weights_path = matches[0] if matches else 'runs/detect/train_fresh/weights/best.pt'

# Print which weights file we are using (so we know the model loaded correctly)
print(f"Loading weights: {weights_path}")

# Load the fine-tuned model using the best weights from training
best = YOLO(weights_path)

# Run validation: the model predicts on val images and we compare to ground truth labels
# This computes mAP, precision, and recall across all 80 object classes
metrics = best.val(data='coco128.yaml')

# Print the four key metrics, formatted to 4 decimal places
print(f"mAP50:      {metrics.box.map50:.4f}")    # Mean Average Precision at IoU 0.50
print(f"mAP50-95:   {metrics.box.map:.4f}")      # Mean Average Precision at IoU 0.50-0.95
print(f"Precision:  {metrics.box.mp:.4f}")       # Overall precision (correct / all detections)
print(f"Recall:     {metrics.box.mr:.4f}")       # Overall recall (found / all real objects)


## Step 5: Run Detection on Sample Images

We test the fine-tuned model on two sample images and display the results  
with bounding boxes drawn around each detected object.

In [ ]:
# cv2 is OpenCV — used for image processing (color conversion here)
import cv2

# os is used for file operations (checking modification time to find newest file)
import os

# glob searches for files — we use it to find the trained model weights
import glob

# YOLO is the model class, Image is for displaying images
from ultralytics import YOLO
from PIL import Image

# Find all best.pt files and sort by modification time (newest first)
# This ensures we use the weights from the most recent training run
matches = sorted(glob.glob('**/best.pt', recursive=True), key=os.path.getmtime, reverse=True)

# Use the newest best.pt file
weights_path = matches[0] if matches else 'runs/detect/train_fresh/weights/best.pt'

# Load the fine-tuned model
model_ft = YOLO(weights_path)

# Print which weights file we are using
print(f"Using fine-tuned weights: {weights_path}")

# Loop over two sample images — bus.jpg (street scene) and zidane.jpg (people)
for url in ['https://ultralytics.com/images/bus.jpg',
            'https://ultralytics.com/images/zidane.jpg']:

    # Extract just the filename from the URL (e.g., 'bus.jpg')
    name = url.split('/')[-1]

    # Run detection on this image — [0] gets the first (only) result
    # conf=0.25 means only show detections with 25%+ confidence
    res = model_ft.predict(url, conf=0.25)[0]

    # res.plot() returns the image with bounding boxes drawn (in BGR color format)
    # cv2.cvtColor converts BGR to RGB — needed because OpenCV uses BGR but display expects RGB
    annotated = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)

    # Display the annotated image with bounding boxes in the notebook
    display(Image.fromarray(annotated))

    # Extract the class names of all detected objects
    # res.boxes contains all detected bounding boxes
    # b.cls is the class ID (integer), res.names[int(b.cls)] converts it to a name string
    names = [res.names[int(b.cls)] for b in res.boxes]

    # Print how many objects were detected and their class names
    print(f"  {name}: {len(res.boxes)} detections — {names}")


## Step 6: Save the Trained Model Weights

Copy the fine-tuned weights (best.pt) to /content/ so they can be  
easily downloaded from the Colab file browser on the left panel.

In [ ]:
# shutil is used for copying files
import shutil

# os and glob are used for finding files
import os, glob

# Path is for handling file paths
from pathlib import Path

# Find all best.pt files, but exclude /content/best.pt itself (avoid self-copy)
matches = sorted(
    [m for m in glob.glob('**/best.pt', recursive=True) if os.path.abspath(m) != '/content/best.pt'],
    key=os.path.getmtime,
    reverse=True  # newest first
)

# Use the newest match, or fall back to default path
weights_path = matches[0] if matches else 'runs/detect/train_fresh/weights/best.pt'

# Destination path — /content/ is the main folder in Colab, easy to find in the file browser
dest = '/content/best.pt'

# Copy the weights file to /content/ (only if source and destination are different)
if os.path.abspath(weights_path) != os.path.abspath(dest):
    shutil.copy2(weights_path, dest)  # copy2 preserves file metadata (timestamps etc.)

# Calculate file size in MB (bytes / 1024 / 1024)
size_mb = Path(dest).stat().st_size / 1024 / 1024

# Print the save location and file size so the user knows where to find it
print(f"Weights saved to: {dest}")
print(f"File size: {size_mb:.1f} MB")
print(f"\nDownload via Colab file browser:")
print(f"  Left panel > Files > best.pt > right-click > Download")
